# Pelajaran 10 - Ejen AI dalam Pengeluaran

Dalam pelajaran ini anda akan mempelajari **corak pengeluaran** untuk ejen AI menggunakan Microsoft Agent Framework (Python). Kami meliputi:

- **Kebolehlihatan** — menambahkan pengukuran masa dan pencatatan kepada interaksi ejen
- **Penilaian** — menggunakan ejen penilai untuk memberi skor kualiti respons
- **Pengurusan kos** — strategi untuk pengoptimuman token dan pemilihan model

Skenario ini ialah sebuah **ejen pelancongan** yang membantu pengguna merancang perjalanan, dengan lapisan pemantauan dan penilaian di atasnya.


## Persediaan


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Pertimbangan Pengeluaran

Memindahkan agen AI dari prototaip ke pengeluaran memerlukan perhatian teliti terhadap tiga tonggak:

1. **Keterlihatan** — Anda memerlukan keterlihatan terhadap apa yang agen lakukan, berapa lama ia mengambil masa, dan alat mana yang dipanggilnya. Tanpa penjejakan dan perekodan log, menyahpepijat isu pengeluaran hampir mustahil.

2. **Penilaian** — Pemeriksaan kualiti automatik memastikan respons agen kekal tepat, lengkap, dan membantu dari masa ke masa. Agen penilai boleh memberikan skor kepada respons mengikut kriteria yang ditetapkan.

3. **Pengurusan Kos** — Penggunaan token memberi kesan langsung kepada kos. Strategi seperti pengoptimuman prompt, pemilihan model, dan caching membantu mengekalkan perbelanjaan di bawah kawalan tanpa mengorbankan kualiti.


## Membina Ejen yang Boleh Dipantau

Kami mentakrifkan alat perjalanan dan membungkus panggilan ejen dengan pengukuran masa supaya kami dapat memantau latensi. Dalam persekitaran pengeluaran, anda akan mengintegrasikannya dengan OpenTelemetry atau backend penjejakan yang serupa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Corak Penilaian

Corak pengeluaran biasa ialah menggunakan agen kedua sebagai seorang **penilai**. Penilai menilai respons agen utama mengikut kriteria yang telah ditetapkan seperti kelengkapan, ketepatan, dan kebermanfaatan.

Ini membolehkan:
- Gerbang kualiti automatik sebelum respons sampai kepada pengguna
- Pengesanan regresi apabila prompt atau model berubah
- Pemantauan berterusan prestasi agen dari masa ke masa


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategi Pengurusan Kos

Mengawal kos adalah penting untuk ejen AI pengeluaran. Berikut adalah strategi utama:

| Strategi | Penerangan |
|---|---|
| **Pengoptimuman arahan** | Pastikan arahan sistem ringkas. Buang konteks berulang untuk mengurangkan token input. |
| **Pemilihan model** | Gunakan model yang lebih kecil dan murah (contohnya GPT-4o-mini) untuk tugas mudah seperti pengkelasan atau pengekstrakan, dan simpan model yang lebih besar untuk penalaran kompleks. |
| **Caching** | Simpan hasil alat dan pertanyaan yang kerap dalam cache untuk mengelakkan panggilan API berulang. |
| **Bajet token** | Tetapkan had `max_tokens` untuk mengelakkan respons yang terlalu panjang secara tidak dijangka. |
| **Pengelompokan** | Kumpulkan beberapa pertanyaan pengguna ke dalam satu panggilan API apabila boleh. |

Dalam amalan, pendekatan berperingkat berfungsi dengan baik: arahkan permintaan yang mudah kepada model yang pantas dan murah dan hanya naikkan pertanyaan yang kompleks kepada model yang lebih berkemampuan.


## Ringkasan

Dalam pelajaran ini anda belajar bagaimana untuk:

1. **Tambah kebolehlihatan** pada interaksi ejen dengan pengukuran masa dan perekodan log, meletakkan asas untuk penjejakan dan pemantauan.
2. **Nilai respon ejen** secara automatik menggunakan ejen penilai yang memberikan skor untuk kelengkapan, ketepatan, dan kegunaan.
3. **Urus kos** melalui pengoptimuman prompt, pemilihan model, caching, dan bajet token.

Corak pengeluaran ini membantu memastikan ejen AI anda boleh dipercayai, dapat diukur, dan menjimatkan kos pada skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan penterjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk memastikan ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi ralat atau ketidaktepatan. Dokumen asal dalam bahasa aslinya hendaklah dianggap sebagai sumber rujukan yang sahih. Untuk maklumat penting, disyorkan mendapatkan terjemahan profesional oleh penterjemah manusia. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsiran yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
